# Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```

### Setup

In [29]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [30]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [31]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [32]:
# TODO: Load environment variables

load_dotenv("config.env")

True

### VectorDB Instance

In [33]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [40]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base="https://openai.vocareum.com/v1",
    model_name="text-embedding-3-small"
)


In [41]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [43]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    existing = collection.get(ids=[doc_id])
    # Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]
    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )


### Verify Semantic Search

Confirm the vector database can be queried for semantic similarity.

In [44]:
# Test query 1: Find racing games on PlayStation
print("=" * 60)
print("TEST QUERY: 'racing game PlayStation'")
print("=" * 60)

results = collection.query(
    query_texts=["racing game PlayStation"],
    n_results=3
)

for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\nResult {i+1}:")
    print(f"  Name: {meta['Name']}")
    print(f"  Platform: {meta['Platform']}")
    print(f"  Year: {meta['YearOfRelease']}")
    print(f"  Genre: {meta['Genre']}")

TEST QUERY: 'racing game PlayStation'

Result 1:
  Name: Gran Turismo 5
  Platform: PlayStation 3
  Year: 2010
  Genre: Racing

Result 2:
  Name: Gran Turismo
  Platform: PlayStation 1
  Year: 1997
  Genre: Racing

Result 3:
  Name: Marvel's Spider-Man
  Platform: PlayStation 4
  Year: 2018
  Genre: Action-adventure


In [45]:
# Test query 2: Find open-world action games
print("=" * 60)
print("TEST QUERY: 'open world action adventure game'")
print("=" * 60)

results = collection.query(
    query_texts=["open world action adventure game"],
    n_results=3
)

for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\nResult {i+1}:")
    print(f"  Name: {meta['Name']}")
    print(f"  Platform: {meta['Platform']}")
    print(f"  Year: {meta['YearOfRelease']}")
    print(f"  Description: {meta['Description'][:100]}...")

print("\n✓ Vector database is working correctly!")

TEST QUERY: 'open world action adventure game'

Result 1:
  Name: Minecraft
  Platform: Xbox One
  Year: 2014
  Description: A sandbox game that allows players to build and explore infinite worlds, fostering creativity and ad...

Result 2:
  Name: Grand Theft Auto: San Andreas
  Platform: PlayStation 2
  Year: 2004
  Description: An expansive open-world game set in the fictional state of San Andreas, following the story of Carl ...

Result 3:
  Name: Marvel's Spider-Man
  Platform: PlayStation 4
  Year: 2018
  Description: An open-world superhero game that lets players swing through New York City as Spider-Man, battling i...

✓ Vector database is working correctly!
